In [ ]:
# Loading the dataset in - since it is so large, it must be done using batch processing

import pandas as pd
import numpy as np
import kagglehub


csv_path = kagglehub.dataset_download(
    "sobhanmoosavi/us-accidents",
    path="US_Accidents_March23.csv",
)

batch_size = 10000
batches = []
batch_num = 0

for batch in pd.read_csv(csv_path, chunksize=batch_size, low_memory=False):
    batches.append(batch)
    batch_num += 1
    if batch_num % 20 == 0:
        print(f"Processed {batch_num * batch_size} rows")

df = pd.concat(batches, ignore_index=True)
print(df.shape)

In [ ]:
# Finding the top 10 cities with the most accidents
df["City"].value_counts().head(10)

In [ ]:
# To trim the dataset, we choose the top 3 cities with the most accidents: Miami, Houston, and Los Angeles
df_trimmed = df[df["City"].isin(["Miami", "Houston", "Los Angeles"])]
df_trimmed.head()

In [ ]:
# select columns to be dropped
columns_dropped = [
    # zero/near-zero variance
    'Turning_Loop',
    'Roundabout',
    'No_Exit',
    'Traffic_Calming',
    'Bump',

    # redundant or irrelevant
    'Country',
    'State',
    'End_Lat',
    'End_Lng',
    'Airport_Code',
    'Weather_Timestamp',
    'County',
    'Timezone',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
    'Wind_Chill(F)',
    'ID',
    'Description',
    'Street',
]

# # create a copy to keep a primary df and apply transformation
d2 = df_trimmed.copy().drop(columns_dropped, axis=1)

print(f"Columns before: {df_trimmed.shape[1]}")
print(f"Columns after: {d2.shape[1]}")
print(f"Remaining columns: {list(d2.columns)}")

In [ ]:
# checking for null values
d2.info()

## Null Value Handling

In [ ]:
# populating null values instead of completely dropping using forward fill (time-series) with a 3 hour gap for similar results (can't use KNN because data too large)

import numpy as np

# sort first
d2 = d2.sort_values(['City', 'Start_Time'])

# change to datetime (valuerror showed to add in format='ISO8601' as a parameter)
d2['Start_Time'] = pd.to_datetime(d2['Start_Time'], format='ISO8601')
d2['End_Time'] = pd.to_datetime(d2['End_Time'], format='ISO8601')

# select weather columns
weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                'Visibility(mi)', 'Wind_Direction', 
                'Wind_Speed(mph)', 'Weather_Condition']

# tweak this threshold as needed, currently 3 hours
max_gap = pd.Timedelta(hours=3)

# forward fill weather values, but only if the time gap between the current row and the last known value is within the max_gap threshold
def ffill_with_time_limit(group, cols, max_gap):
    for col in cols:
        filled = group[col].ffill()

        # calculate time diff between each row and the last non-null row
        last_valid_time = group['Start_Time'].where(group[col].notna()).ffill()
        time_gap = group['Start_Time'] - last_valid_time

        # only accept the fill if the gap is within threshold
        group[col] = filled.where(time_gap <= max_gap, other=np.nan)
    return group

# apply the function to each city group (done by city to avoid filling across different cities which may have different weather conditions)
city_groups = []
for city, group in d2.groupby('City'):
    filled_group = ffill_with_time_limit(group, weather_cols, max_gap)
    city_groups.append(filled_group)

# concatenate the groups back together
d2 = pd.concat(city_groups).sort_values(['City', 'Start_Time'])

# precipitation still just fills with 0
d2['Precipitation(in)'] = d2['Precipitation(in)'].fillna(0)

# check remaining nulls
print(d2.isnull().sum())

In [ ]:
# drop the remaining null columns
d2 = d2.dropna()
# reset the index for simplicity
d2.reset_index(drop=True)

## Feature Engineering

In [ ]:
# Adding a duration of accidents in minutes

d2['Duration_Minutes'] = (d2['End_Time'] - d2['Start_Time']).dt.total_seconds() / 60

# Changing Start_Time and End_Time to separate date and time columns 
d2['Start_Date'] = d2['Start_Time'].dt.date
d2['Start_Clock_Time'] = d2['Start_Time'].dt.time

d2["End_Date"] = d2["End_Time"].dt.date
d2["End_Clock_Time"] = d2["End_Time"].dt.time

In [ ]:
import holidays

# create a US holidays object
us_holidays = holidays.UnitedStates()
# create a Holiday column that indicates whether the date is a holiday
d2['Holiday'] = d2['Start_Date'].apply(lambda x: x in us_holidays).astype(int)
d2['Holiday_Name'] = d2['Start_Time'].dt.date.apply(lambda x: us_holidays.get(x))
d2["Holiday_Name"].fillna("Not Holiday", inplace=True)

In [ ]:
d2[d2['Holiday'] == 1]